# Fetch 7 Days of News with Specific Keywords

In [1]:
import csv
import json
import os
import time
from urllib.parse import urlencode
from urllib.request import Request, urlopen

BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"
OUTPUT_DIR = "data"
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "gdelt_english_news.csv")

FIELDNAMES = [
    "seendate",
    "title",
    "url",
    "url_mobile",
    "domain",
    "language",
    "sourcecountry",
    "socialimage",
]

In [12]:
def load_seen_urls(csv_path=OUTPUT_CSV):
    if not os.path.exists(csv_path):
        return set()
    with open(csv_path, newline="", encoding="utf-8") as handle:
        return {row.get("url") for row in csv.DictReader(handle) if row.get("url")}

def append_articles_to_csv(articles, csv_path=OUTPUT_CSV):
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    file_exists = os.path.exists(csv_path)
    with open(csv_path, "a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=FIELDNAMES)
        if not file_exists:
            writer.writeheader()
        for article in articles:
            row = {field: article.get(field, "") for field in FIELDNAMES}
            writer.writerow(row)

def append_new_articles(articles, seen_urls, csv_path=OUTPUT_CSV):
    new_articles = []
    for article in articles:
        url = article.get("url")
        if url and url not in seen_urls:
            seen_urls.add(url)
            new_articles.append(article)
    append_articles_to_csv(new_articles, csv_path=csv_path)
    return len(new_articles)

In [13]:
# Fetch 7 days of news with specific keywords to get more unique articles

queries = [
    "sourcelang:english AI",
    "sourcelang:english cryptocurrency",
    "sourcelang:english software",
    "sourcelang:english cybersecurity",
    "sourcelang:english stock market",
    "sourcelang:english economy",
    "sourcelang:english startup",
    "sourcelang:english election",
    "sourcelang:english government",
    "sourcelang:english policy",
    "sourcelang:english football",
    "sourcelang:english basketball",
    "sourcelang:english cricket",
    "sourcelang:english medical",
    "sourcelang:english vaccine",
    "sourcelang:english fitness",
    "sourcelang:english climate",
    "sourcelang:english space",
    "sourcelang:english research",
    "sourcelang:english movie",
    "sourcelang:english music",
    "sourcelang:english celebrity",
]

def fetch_with_query(query, timespan="7days", maxrecords=250):
    params = {
        "query": query,
        "mode": "artlist",
        "maxrecords": maxrecords,
        "timespan": timespan,
        "format": "json",
        "sort": "datedesc",
    }
    url = f"{BASE_URL}?{urlencode(params)}"
    req = Request(url, headers={"User-Agent": "Mozilla/5.0", "Accept": "application/json"})
    with urlopen(req, timeout=30) as response:
        text = response.read().decode("utf-8", errors="replace")
    data = json.loads(text)
    return data.get("articles", [])

seen_urls = load_seen_urls()
total_new = 0

print(f"Fetching {len(queries)} different topics from past 7 days...\n")

for i, query in enumerate(queries, 1):
    try:
        keyword = query.split()[-1]
        articles = fetch_with_query(query, timespan="7days", maxrecords=250)
        new_count = append_new_articles(articles, seen_urls)
        total_new += new_count
        print(f"{i:2}. {keyword:15} - {len(articles)} fetched, {new_count:3} new")
        time.sleep(1)
    except Exception as e:
        print(f"{i:2}. Error: {e}")

print(f"\n✓ Total new articles added: {total_new}")
print(f"✓ Total unique articles: {len(seen_urls)}")

Fetching 22 different topics from past 7 days...

 1. Error: Expecting value: line 1 column 1 (char 0)
 2. cryptocurrency  - 250 fetched, 245 new
 3. software        - 250 fetched, 227 new
 4. cybersecurity   - 250 fetched, 213 new
 5. market          - 250 fetched, 177 new
 6. economy         - 250 fetched, 201 new
 7. startup         - 250 fetched, 214 new
 8. election        - 250 fetched, 168 new
 9. government      - 250 fetched, 154 new
10. policy          - 250 fetched, 122 new
11. football        - 250 fetched, 228 new
12. basketball      - 250 fetched, 205 new
13. cricket         - 250 fetched, 236 new
14. medical         - 250 fetched, 186 new
15. vaccine         - 250 fetched, 241 new
16. fitness         - 250 fetched, 222 new
17. climate         - 250 fetched, 199 new
18. space           - 250 fetched, 128 new
19. research        - 250 fetched, 104 new
20. movie           - 250 fetched, 221 new
21. music           - 250 fetched, 156 new
22. celebrity       - 250 fetched, 21

In [14]:
# Check final dataset size
seen = load_seen_urls()
print(f"Total unique articles in dataset: {len(seen):,}")

Total unique articles in dataset: 4,308
